# Install Necessay Libraries

In [1]:
!pip install -qU langchain langchain-text-splitters langchain-cohere langchain-qdrant langchain-community
!pip install -qU qdrant-client beautifulsoup4 lxml
!pip install -qU pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.1/127.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 126.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

# Import Libraries


In [2]:
import os #read API
import cohere

from google.colab import userdata
from langchain_community.document_loaders import WebBaseLoader , PyPDFLoader # Fetches and parses web pages into Documents
from langchain_text_splitters import RecursiveCharacterTextSplitter #Splits long Documents into smaller chunks
from langchain_cohere import CohereEmbeddings,ChatCohere
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

/tmp/ipykernel_4906/2196447858.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader , PyPDFLoader # Fetches and parses web pages into Documents


# Config

In [3]:
#Load Secrets
os.environ['COHERE_API_KEY'] = userdata.get('COHERE_API_KEY')
os.environ['QDRANT_API_KEY'] = userdata.get('QDRANT_API_KEY')
os.environ['QDRANT_URL'] = userdata.get('QDRANT_URL')
print('Api Key Loaded!')

#configuration
cohere_embed_model = 'embed-english-v3.0'
cohere_chat_model = 'command-r-08-2024'
qdrant_collection = 'RAG_web-pdf'

#chunks the sources
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 4
print('Configuration Set!')

Api Key Loaded!
Configuration Set!


# Load Document

In [4]:

print('LOADING & PROCESSING DOCUMENT')
print('='*60)

urls = [
    'https://docs.cohere.com/docs/embeddings',
    'https://docs.langchain.com/oss/python/langchain/retrieval'

    ]

# pdf = [
#     '/content/Chapter_4_v9.0[1].pdf'
# ]

print('Loading Document')
load_url = WebBaseLoader(web_paths=urls)
load_pdf = PyPDFLoader('/content/Chapter_4_v9.0[1].pdf')

load_docs = load_url.load() + load_pdf.load()
print(f'Loaded {len(load_docs)} documents')

print('Split into Chunks')
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

splits = text_spliter.split_documents(load_docs)
print(f'Split into {len(splits)} chunks')
print(f'Preview: {splits[0].page_content[:200]}')

# Check what sources were actually loaded
# sources = set([doc.metadata.get('source', 'unknown') for doc in load_docs])
# print("Sources found:")
# for source in sources:
#     print(f"  → {source}")

# web_docs = load_url.load()
# pdf_docs = load_pdf.load()
# print(f"Web documents: {len(web_docs)}")
# print(f"PDF pages: {len(pdf_docs)}")
# print(f"Total: {len(load_docs)}")

LOADING & PROCESSING DOCUMENT
Loading Document
Loaded 104 documents
Split into Chunks
Split into 147 chunks
Preview: Introduction to Embeddings at Cohere | CohereFor AI agents: a documentation index is available at the root level at /llms.txt and /llms-full.txt. Append /llms.txt to any URL for a page-level index, or


# Create Vector

In [5]:
print('=' *60)
print('Creating Vector Store')

print('Initializing Embeddings')
embeddings = CohereEmbeddings(
    cohere_api_key= os.getenv('COHERE_API_KEY'),
    model = cohere_embed_model
)
#embeddings.input_type = "search_document"
test_vector = embeddings.embed_query('test')
vector_size = len(test_vector)
print(f'Vector Size: {vector_size}')

print('Connecting to Qdrant VectoreStore')
qdrant = QdrantClient(
    url=os.getenv('QDRANT_URL'),
    api_key=os.getenv('QDRANT_API_KEY')
)

existing_collection = [c.name for c in qdrant.get_collections().collections]
print(existing_collection) #empty , cluster have no collection yet

if qdrant_collection not in existing_collection:
    print('Collection not found - Creating Collection')

    qdrant.create_collection(
        collection_name = qdrant_collection,
        vectors_config= VectorParams(size=vector_size, distance=Distance.COSINE)
    )

    vectorStore = QdrantVectorStore.from_documents(
        documents = splits,
        embedding = embeddings,
        url = os.getenv('QDRANT_URL'),
        api_key = os.getenv('QDRANT_API_KEY'),
        collection_name = qdrant_collection
    )

    print(f'Stored {len(splits)} chunks!')

else:
  print('collection found - connecting')
  vectorStore = QdrantVectorStore.from_existing_collection(
      embedding = embeddings,
      collection_name = qdrant_collection,
      url = os.getenv('QDRANT_URL'),
      api_key = os.getenv('QDRANT_API_KEY')
  )
  print('connected to existing collection')




Creating Vector Store
Initializing Embeddings
Vector Size: 1024
Connecting to Qdrant VectoreStore
['RAG_web-pdf']
collection found - connecting
connected to existing collection


In [ ]:
#check url
#print(repr(os.getenv('QDRANT_URL')))

'https://a673ac83-b4f2-4ce7-9c72-a5384de9aabc.us-east-2-0.aws.cloud.qdrant.io'


# Building Rag System

In [6]:
llm = ChatCohere(
    cohere_api_key = os.getenv('COHERE_API_KEY'),
    model = cohere_chat_model,
    temperature= 0.7,
    max_tokens = 512

)


print('='*60)

def queryRag(question: str):
  print(f'Question : {question}')
  print('='*60)

  #retrieve relevant chunks from vectorstore

  docs = vectorStore.similarity_search(
      query = question,
      k = TOP_K
  )

  #join chunks into one context string

  context = '\n\n'.join([doc.page_content for doc in docs])

  #generate answer

  print(" Generating answer with Cohere...")
  prompt = f'''Based on the following context, answer the question clearly and concisely.


Context:
{context}

Question: {question}


Answer:'''
  response = llm.invoke(prompt)
  answer = response.content.strip()
  print("\nANSWER:")
  print("-"*60)
  print(answer)
  print("-"*60)
  print(f"Based on {len(docs)} chunks")





In [ ]:
queryRag("What are Cohere embeddings?")

Question : What are Cohere embeddings?
 Generating answer with Cohere...

ANSWER:
------------------------------------------------------------
Cohere embeddings are a platform and set of models that create vector representations of various data types, including text, images, and audio. These embeddings can be used for a range of tasks such as semantic search, retrieval, and classification. The platform supports compression and allows users to specify output dimensions and embedding types.
------------------------------------------------------------
Based on 4 chunks


In [ ]:
queryRag('what is forwarding?')

Question : what is forwarding?
 Generating answer with Cohere...

ANSWER:
------------------------------------------------------------
Forwarding is the process of moving packets from a router's input link to the appropriate router output link. It can be likened to getting through a single interchange during a trip.
------------------------------------------------------------
Based on 4 chunks


In [ ]:
queryRag('data plane & control plane explain?')

Question : data plane & control plane explain?
 Generating answer with Cohere...

ANSWER:
------------------------------------------------------------
The data plane is responsible for the local, per-router function and determines how a datagram is forwarded from the router's input port to its output port. It is a fundamental part of the network layer and handles the actual data transmission. 

The control plane, on the other hand, is concerned with network-wide logic and determines how a datagram is routed among routers along the end-to-end path from the source host to the destination host. It provides the overall routing strategy and control over the network.
------------------------------------------------------------
Based on 4 chunks
